In [0]:
%run ./create_table_utillity

#### Reading unique_carriers data using autoloader

In [0]:
from pyspark.sql.functions import to_date, current_timestamp

# Reading data
df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", "/dbfs/FileStore/tables/schema/cancellation")
    .load("/mnt/geeks/rw_adls/Cancellation/")
)
df = df.withColumn("Date_Part", to_date(current_timestamp()))
display(df)

In [0]:
df_base = df.selectExpr(
    "replace(Code,'\"','') as code",
    "replace(Description,'\"','') as description",
    "to_date(Date_Part,'yyyy-MM-dd') as Date_Part"
)

display(df_base)

In [0]:
# Writing data
df_base.writeStream.trigger(once=True).format("delta").option(
    "checkpointLocation", "/dbfs/FileStore/tables/checkpointLocation/cancellation"
).start("/mnt/geeks/cld_adls/cancellation")

#### Creating delata table on the data

In [0]:

f_delta_cleansed_load('cancellation', 'abfss://cleansed@geekadlsstoragesink.dfs.core.windows.net/cancellation', 'cleansed_geekcoders')

In [0]:
%sql
select * from cleansed_geekcoders.cancellation;